In [1]:
"""
DSC2107: Data Mining & Wrangling
Warm-up Exercise (Day 1): books.toscrape.com
Target: Extract all ~1,000 books with pagination and detail page inspection.
"""

from datetime import datetime
import re
import time
from urllib.parse import urljoin
import pandas as pd
import requests
from bs4 import BeautifulSoup

# ==========================================
# 1. CONFIGURATION & ETHICAL HEADERS
# ==========================================
BASE_URL = "https://books.toscrape.com/catalogue/"
START_URL = "https://books.toscrape.com/catalogue/page-1.html"

# UCU Assignment Rule: Identify yourself clearly
HEADERS = {
    "User-Agent": "DSC2107-Course-Project (student_email@ucu.ac.ug)"
}

RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def fetch_page(url, retries=3, delay=2):
    """
    Fetches a web page with retry logic and polite back-off.
    """
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            if response.status_code == 200:
                return response
            elif response.status_code == 404:
                print(f"[404] Page not found: {url}")
                return None
            else:
                print(f"[Warning] Status {response.status_code} on attempt {attempt}. Retrying...")
        except requests.RequestException as e:
            print(f"[Error] Request failed on attempt {attempt}: {e}")
        
        time.sleep(delay * attempt)
    return None

def extract_detail_page(detail_url):
    """
    Visits the book's individual detail page to grab:
    Category, UPC, Exact Stock Count, and Product Description.
    """
    resp = fetch_page(detail_url)
    if not resp:
        return {}

    soup = BeautifulSoup(resp.text, "html.parser")
    details = {}

    # 1. Category (from breadcrumb: Home > Books > [Category] > Title)
    breadcrumb = soup.find("ul", class_="breadcrumb")
    if breadcrumb:
        crumbs = breadcrumb.find_all("li")
        if len(crumbs) >= 3:
            details["category"] = crumbs[2].text.strip()
        else:
            details["category"] = "Unknown"
    else:
        details["category"] = "Unknown"

    # 2. Product Description
    sub_header = soup.find("div", id="product_description")
    if sub_header:
        p_desc = sub_header.find_next_sibling("p")
        details["description"] = p_desc.text.strip() if p_desc else ""
    else:
        details["description"] = ""

    # 3. Product Information Table (UPC, Tax, Exact Stock count)
    table = soup.find("table", class_="table-striped")
    if table:
        rows = table.find_all("tr")
        table_data = {row.th.text.strip(): row.td.text.strip() for row in rows}
        
        details["upc"] = table_data.get("UPC", "")
        
        # Parse exact numeric stock from e.g. "In stock (22 available)"
        avail_text = table_data.get("Availability", "")
        stock_match = re.search(r"\((\d+)\s+available\)", avail_text)
        details["stock_quantity"] = int(stock_match.group(1)) if stock_match else 0
    else:
        details["upc"] = ""
        details["stock_quantity"] = 0

    return details

# ==========================================
# 3. MAIN SCRAPING ENGINE
# ==========================================
def scrape_all_books():
    all_books = []
    current_url = START_URL
    page_num = 1
    
    start_time = datetime.now()
    print(f"=== Starting books.toscrape.com Warm-Up at {start_time.strftime('%Y-%m-%d %H:%M:%S')} ===")

    while current_url:
        print(f"\n[Page {page_num}] Fetching listing: {current_url}")
        response = fetch_page(current_url)
        
        if not response:
            print("Failed to retrieve listing page. Stopping.")
            break

        soup = BeautifulSoup(response.text, "html.parser")
        book_cards = soup.find_all("article", class_="product_pod")
        
        print(f"-> Found {len(book_cards)} books on page {page_num}.")

        for card in book_cards:
            try:
                # 1. Title
                title = card.h3.a["title"]

                # 2. Price (strip '£' symbol and convert to float)
                price_text = card.find("p", class_="price_color").text
                price = float(re.sub(r"[^\d.]", "", price_text))

                # 3. Rating (from CSS class)
                rating_classes = card.find("p", class_="star-rating")["class"]
                rating_word = [c for c in rating_classes if c != "star-rating"][0]
                rating = RATING_MAP.get(rating_word, 0)

                # 4. Detail page link
                rel_url = card.h3.a["href"]
                detail_url = urljoin(current_url, rel_url)

                # 5. Visit Detail Page (to satisfy "detail pages" requirement)
                # Polite micro-pause before visiting detail page
                time.sleep(0.5)
                detail_info = extract_detail_page(detail_url)

                # Combine all attributes
                book_record = {
                    "upc": detail_info.get("upc", ""),
                    "title": title,
                    "category": detail_info.get("category", ""),
                    "price_gbp": price,
                    "rating_stars": rating,
                    "stock_quantity": detail_info.get("stock_quantity", 0),
                    "product_url": detail_url,
                    "description": detail_info.get("description", ""),
                    "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }

                all_books.append(book_record)
                print(f"   ✓ Scraped: {title[:40]}... (Stock: {book_record['stock_quantity']}, Rating: {rating}★)")

            except Exception as e:
                print(f"   ✗ Error parsing book card: {e}")
                continue

        # Check for next page
        next_button = soup.find("li", class_="next")
        if next_button:
            next_page_rel = next_button.a["href"]
            current_url = urljoin(current_url, next_page_rel)
            page_num += 1
            time.sleep(1.5)  # Polite delay between listing pages
        else:
            print("\n-> Reached the last page. Scraping finished!")
            current_url = None

    # ==========================================
    # 4. SAVE RAW EVIDENCE & PRINT SUMMARY
    # ==========================================
    print("\n=== Generating Dataset Summary ===")
    df = pd.DataFrame(all_books)
    
    # Save to CSV
    output_filename = "books_warmup.csv"
    df.to_csv(output_filename, index=False, encoding="utf-8")
    
    print(f"Total books scraped: {len(df)}")
    print(f"Saved dataset to: {output_filename}")
    print("\nFirst 3 rows preview:")
    print(df[["upc", "title", "category", "price_gbp", "rating_stars", "stock_quantity"]].head(3))
    
    elapsed = datetime.now() - start_time
    print(f"\nExecution finished in: {elapsed}")

if __name__ == "__main__":
    scrape_all_books()

=== Starting books.toscrape.com Warm-Up at 2026-10-14 15:10:20 ===

[Page 1] Fetching listing: https://books.toscrape.com/catalogue/page-1.html
-> Found 20 books on page 1.
   ✓ Scraped: A Light in the Attic... (Stock: 22, Rating: 3★)
   ✓ Scraped: Tipping the Velvet... (Stock: 20, Rating: 1★)
   ✓ Scraped: Soumission... (Stock: 20, Rating: 1★)
   ✓ Scraped: Sharp Objects... (Stock: 20, Rating: 4★)
   ✓ Scraped: Sapiens: A Brief History of Humankind... (Stock: 20, Rating: 5★)
   ✓ Scraped: The Requiem Red... (Stock: 19, Rating: 1★)
   ✓ Scraped: The Dirty Little Secrets of Getting Your... (Stock: 19, Rating: 4★)
   ✓ Scraped: The Coming Woman: A Novel Based on the L... (Stock: 19, Rating: 3★)
   ✓ Scraped: The Boys in the Boat: Nine Americans and... (Stock: 19, Rating: 4★)
   ✓ Scraped: The Black Maria... (Stock: 19, Rating: 1★)
   ✓ Scraped: Starving Hearts (Triangular Trade Trilog... (Stock: 19, Rating: 2★)
   ✓ Scraped: Shakespeare's Sonnets... (Stock: 19, Rating: 4★)
   ✓ Scraped: 